# 04 | The Full Transformer Block
## Sprint 4 — Day 2

**Previous notebook:** `03` — multi-head attention implemented and tested.
We now have all the pieces. This notebook assembles them.

---

A single Transformer encoder block consists of:

1. **Multi-Head Attention** (built in `03`)
2. **Add & Norm** — a residual connection followed by Layer Normalisation
3. **Feed-Forward Network** — two linear layers with a ReLU in between
4. **Add & Norm** — again

We will also cover *why* each of these exists. Residual connections are not
optional — they are what makes deep Transformers trainable. Layer Norm is not
the same as Batch Norm, and the difference matters.

---

**By the end of this notebook you will have:**
- Implemented Layer Normalisation from scratch
- Built the Feed-Forward sublayer
- Assembled a complete, reusable `TransformerBlock` module
- Verified that output shape matches input shape end-to-end

---

*Next notebook: `05_building_a_tiny_gpt.ipynb`*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

d_model     = 64
num_heads   = 8
d_ff        = 256
max_seq_len = 50
vocab_size  = 256
batch_size  = 2
seq_len     = 10
dropout_rate = 0.1

assert d_model % num_heads == 0

def positional_encoding(max_seq_len, d_model):
    PE = torch.zeros(max_seq_len, d_model)
    position = torch.arange(0, max_seq_len).unsqueeze(1).float()
    div_term = torch.pow(10000.0, torch.arange(0, d_model, 2).float() / d_model)
    PE[:, 0::2] = torch.sin(position / div_term)
    PE[:, 1::2] = torch.cos(position / div_term)
    return PE

class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pe = positional_encoding(max_seq_len, d_model)
    def forward(self, x):
        return self.embedding(x) + self.pe[:x.shape[1], :]

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, V), weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        B, T, _ = x.shape
        Q = self.W_Q(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        attn_out, weights = scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_O(attn_out), weights

embed_layer = InputEmbedding(vocab_size, d_model, max_seq_len)

print("Setup complete.")
print(f"d_model={d_model} | num_heads={num_heads} | d_ff={d_ff}")